
# Chapter 1: Arithmetic Functions and Multiplicatives

Arithmetic functions such as the divisor function, Euler’s totient, and the Möbius function form the backbone of multiplicative number theory.  In this chapter we follow the recipe from the book outline【155477964815303†L35-L41】—identify active mechanisms, implement them in SageMath, and explain the conceptual picture—for the first entry in the chapter’s table: the divisor function $d(n)$ (OEIS A000005)【155477964815303†L45-L53】.



## Sequence Analyst – Mechanisms & DAG for $d(n)$

**Active mechanisms**

- **ZETA (Primary):** The Dirichlet series of the divisor function is $\zeta(s)^2=\sum_{n\ge1}
rac{d(n)}{n^s}$【975973847736663†L283-L289】.  Euler products show that $d(n)$ is multiplicative and encode the prime factorization of $n$.
- **MOD (Emergent):** Expanding the two‐variable generating function $\sum_{n\ge1}d(n)q^n$ gives a $q$‐series $\sum_{k\ge1}
rac{q^k}{1-q^k}$ that hints at modular forms.

**Base–Bridge–Emergence DAG**

- **Base:** The combinatorial definition of $d(n)$ counts the number of divisors of $n$.  For $n=p_1^{e_1}\cdots p_r^{e_r}$ it equals $(e_1+1)\cdots(e_r+1)$.
- **Bridge:** Viewing $d(n)$ as the Dirichlet convolution of the constant function $1$ with itself ($1*1=d$) exposes its multiplicativity.  The corresponding Dirichlet series factorizes as $\zeta(s)^2$【975973847736663†L283-L289】.
- **Emergence:** Summing $d(n)q^n$ across all $n$ yields $\sum_{k\ge1}
rac{q^k}{1-q^k}$, a $q$‐series reminiscent of Eisenstein series and modular forms.  This emergent mechanism connects arithmetic functions to analysis on the upper half‐plane.


In [ ]:

from math import gcd, log, sqrt
from functools import reduce

# Prime generation helper
def is_prime(n):
    if n < 2:
        return False
    if n in (2,3):
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    w = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += w
        w = 6 - w
    return True

def prime_factors(n):
    i = 2
    factors = {}
    while i * i <= n:
        while n % i == 0:
            factors[i] = factors.get(i, 0) + 1
            n //= i
        i += 1
    if n > 1:
        factors[n] = factors.get(n, 0) + 1
    return factors

# Number of divisors
def d(n):
    pf = prime_factors(n)
    result = 1
    for e in pf.values():
        result *= (e + 1)
    return result

# Sum of divisors

def sigma(n):
    pf = prime_factors(n)
    s = 1
    for p, e in pf.items():
        s *= (p**(e+1) - 1) // (p - 1)
    return s

def s_proper(n):
    return sigma(n) - n

# Totient function

def phi(n):
    pf = prime_factors(n)
    result = n
    for p in pf:
        result = result // p * (p - 1)
    return result

# Number of distinct prime factors

def omega(n):
    return len(prime_factors(n))

# Total number of prime factors (with multiplicity)

def Omega(n):
    return sum(prime_factors(n).values())

# Möbius function

def mu(n):
    pf = prime_factors(n)
    for e in pf.values():
        if e > 1:
            return 0
    return -1 if len(pf) % 2 else 1

# Dirichlet convolution

def dirichlet_convolution(f, g, n):
    total = 0
    for d in range(1, n+1):
        if n % d == 0:
            total += f(d) * g(n//d)
    return total

# Prime counting function pi(n)

def primes_upto(n):
    primes = []
    for k in range(2, n+1):
        if is_prime(k):
            primes.append(k)
    return primes

def pi_fn(n):
    return len(primes_upto(n))

# Primorials

def primorial(n):
    ps = primes_upto(10000)  # generate enough primes for our range
    prod = 1
    for i in range(n):
        prod *= ps[i]
    return prod


In [ ]:

print("="*70)
print("DIVISOR FUNCTION d(n)")
print("="*70)


In [ ]:

# --- 1. Direct Computation ---
print("
1. First 20 values of d(n):")
# Use the divisor function sigma(n,0) which counts the number of divisors
vals = [sigma(n,0) for n in range(1,21)]
print(vals)


In [ ]:

# --- 2. Multiplicativity Check ---
print("
2. Multiplicativity: d(ab) = d(a)d(b) when gcd(a,b)=1")
# Test random coprime pairs
import random
from math import gcd

ok = True
for _ in range(10):
    a = random.randint(1,50)
    b = random.randint(1,50)
    if gcd(a,b) == 1:
        lhs = sigma(a*b,0)
        rhs = sigma(a,0) * sigma(b,0)
        if lhs != rhs:
            ok = False
            print(f"   Failure at a={a}, b={b}: d(ab)={lhs}, d(a)d(b)={rhs}")
print(f"   Multiplicativity holds? {ok}")


In [ ]:

# --- 3. Dirichlet Series: ζ(s)^2 versus Σ d(n)/n^s ---
print("
3. Dirichlet series approximation for s=2:")

# Choose s and partial sum cutoff N
s = 2
N = 200

# Compute partial sum of d(n)/n^s
partial_sum = sum(sigma(n,0) / n^s for n in range(1, N+1))

# Compute zeta(s)^2 using zeta function
zeta_val = zeta(s)
product_val = zeta_val^2

print(f"   Partial sum Σ_{1≤n≤{N}} d(n)/n^{s} ≈ {partial_sum.n(digits=12)}")
print(f"   ζ({s})^2 = {product_val.n(digits=12)}")
print(f"   Approximation error: {(product_val - partial_sum).abs().n(digits=12)}")


In [ ]:

# --- 4. Prime Power Property ---
print("
4. d(p^k) = k+1 for prime p and exponent k")

primes_list = [2,3,5,7,11]
exponents = [1,2,3,4]
all_good = True
for p in primes_list:
    for k in exponents:
        n = p^k
        lhs = sigma(n,0)
        rhs = k + 1
        if lhs != rhs:
            all_good = False
            print(f"   Mismatch at p={p}, k={k}: d(p^k)={lhs}, expected {rhs}")
print(f"   Property holds for tested primes/exponents? {all_good}")


In [ ]:

# --- 5. Dirichlet Convolution: d = 1 * 1 ---
print("
5. Dirichlet convolution: verify d(n) = (1*1)(n)")

# Define constant function 1 on positive integers
def f_const_one(n):
    return 1

# Dirichlet convolution of f and g evaluated at n

def dirichlet_convolution(f, g, n):
    return sum(f(d) * g(n//d) for d in divisors(n))

# Check for n up to 20
valid = True
for n in range(1, 21):
    conv = dirichlet_convolution(f_const_one, f_const_one, n)
    if conv != sigma(n,0):
        valid = False
        print(f"   Failure at n={n}: convolution {conv}, d(n)={sigma(n,0)}")
print(f"   Dirichlet convolution identity holds for n≤20? {valid}")


In [ ]:

# --- 6. q-Series Generating Function ---
print("
6. q-Series: Σ d(n) q^n = Σ_{k≥1} q^k/(1 - q^k)")

# Power series ring to expand both sides
R.<q> = PowerSeriesRing(QQ, default_prec=15)

# Left-hand side: sum d(n) q^n up to q^14
lhs_series = sum(sigma(n,0)*q^n for n in range(1,15))

# Right-hand side: sum over k of q^k/(1 - q^k)
rhs_series = sum(q^k/(1 - q^k) for k in range(1,8))  # finite sum approximation

print(f"   LHS coefficients: {lhs_series.list()}")
print(f"   RHS coefficients (approx): {rhs_series.list()}")

# Check first few coefficients match
def lists_match(a,b,cut):
    return all(a[i] == b[i] for i in range(cut))

print(f"   Do the first 10 coefficients agree? {lists_match(lhs_series.list(), rhs_series.list(), 10)}")



## Mathematical Expositor – Conceptual Explanations

1. **Direct computation:** Evaluating $d(n)$ for the first few $n$ reveals how the function behaves on prime powers and products.  Composite numbers with many small prime factors have larger $d(n)$.

2. **Multiplicativity:** The test confirms that $d$ is multiplicative: if $a$ and $b$ are coprime then $d(ab)=d(a)d(b)$.  This comes from the factorization of divisors over coprime components, and it is the cornerstone of multiplicative number theory.

3. **Dirichlet series:** Numerically comparing $\sum d(n)n^{-s}$ to $\zeta(s)^2$ for $s=2$ shows rapid convergence, confirming the identity $\zeta(s)^2=\sum_{n\ge1}d(n)n^{-s}$【975973847736663†L283-L289】.  The product expansion encodes the prime‐factor multiplicative structure.

4. **Prime power property:** For a prime $p$ and exponent $k$, $d(p^k)=k+1$.  This follows because the divisors are exactly $p^0, p^1,\dots,p^k$, giving $k+1$ distinct divisors.  When $n$ factorizes into distinct primes with exponents $e_i$, $d(n)=(e_1+1)\cdots(e_r+1)$.

5. **Dirichlet convolution:** The identity $d=1st 1$ expresses $d$ as the convolution of two simple constant functions.  Convolutions correspond to products of Dirichlet series; thus $D_{1}(s)^2=\zeta(s)^2$ matches the Dirichlet series of $d$.

6. **q-Series generating function:** The generating function $\sum_{n\ge1}d(n)q^n$ can be written as $\sum_{k\ge1}
rac{q^k}{1-q^k}$ by grouping divisors according to their contribution.  Expanding both sides as formal power series shows that coefficients coincide for the first several terms, hinting at connections with modular forms.

These demonstrations illustrate how a simple arithmetic function links zeta functions, Dirichlet convolutions and $q$‐series.  The mechanisms identified in the DAG come to life through explicit computation.



## Sequence Analyst – Mechanisms & DAG for A000010 – Totient function

* **Base:** counts integers ≤ n that are coprime to n.
* **Bridge:** φ = μ * id; Dirichlet series ζ(s-1)/ζ(s).
* **Emergence:** Euler’s theorem, RSA, normal orders.


In [ ]:
print("="*70)
print("A000010 – TOTIENT FUNCTION")


In [ ]:
# --- φ(n) Direct Computation ---
N = 20
phi_values = [phi(k) for k in range(1, N+1)]
print("φ(n) for n=1..20:", phi_values)


In [ ]:
# --- φ(n) Multiplicativity Check ---
# Check φ(ab) = φ(a)*φ(b) for coprime a,b
for a in range(1, 11):
    for b in range(1, 11):
        if gcd(a, b) == 1:
            if phi(a*b) != phi(a)*phi(b):
                print("Multiplicativity failed at", a, b)
print("Multiplicativity holds for a,b ≤ 10")


In [ ]:
# --- φ(n) Dirichlet Series Approximation ---
# Compare ζ(s-1)/ζ(s) with Σ φ(n)/n^s for s=3
import cmath
s = 3
# compute partial sums
partial_sum = sum(phi(n) / (n**s) for n in range(1, 2000))
# approximate ζ(s) and ζ(s-1)
# using Euler product truncated: not very accurate but gives idea
import mpmath as mp
mp.dps = 30
zeta = mp.zeta
approx = zeta(s-1) / zeta(s)
print("Σ φ(n)/n^s (partial, 2000 terms) ≈", partial_sum)
print("ζ(s-1)/ζ(s) ≈", approx)


In [ ]:
# --- φ(n) Möbius Inversion Check ---
# Verify that μ * id = φ
for n in range(1, 50):
    conv = dirichlet_convolution(mu, lambda m: m, n)
    if conv != phi(n):
        print("Möbius inversion failed at", n)
print("Möbius inversion holds for n ≤ 50")


## Mathematical Expositor – A000010 – Totient function

Explain the conceptual meaning of φ(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A000203 – Sum of divisors

* **Base:** sums all positive divisors of n.
* **Bridge:** σ = id * 1; Dirichlet series ζ(s)ζ(s-1).
* **Emergence:** classification via s(n)=σ(n)-n; growth estimates.


In [ ]:
print("="*70)
print("A000203 – SUM OF DIVISORS")


In [ ]:
# --- σ(n) Direct Computation ---
N=20
sigma_values=[sigma(k) for k in range(1,N+1)]
print("σ(n) for n=1..20:", sigma_values)


In [ ]:
# --- σ(n) Multiplicativity Check ---
for a in range(1,11):
    for b in range(1,11):
        if gcd(a,b)==1:
            if sigma(a*b) != sigma(a)*sigma(b):
                print("Multiplicativity failed at",a,b)
print("Multiplicativity holds for a,b ≤ 10")


In [ ]:
# --- σ(n) Dirichlet Series Approximation ---
s = 3
partial_sum = sum(sigma(n)/(n**s) for n in range(1,2000))
import mpmath as mp
mp.dps = 30
approx = mp.zeta(s)*mp.zeta(s-1)
print("Σ σ(n)/n^s (partial, 2000 terms) ≈", partial_sum)
print("ζ(s)ζ(s-1) ≈", approx)


In [ ]:
# --- σ(n) Convolution Check ---
# σ = id * 1
for n in range(1,50):
    conv = dirichlet_convolution(lambda m: m, lambda m: 1, n)
    if conv != sigma(n):
        print("Convolution failed at", n)
print("σ = id * 1 holds for n ≤ 50")



## Mathematical Expositor – A000203 – Sum of divisors

Explain the conceptual meaning of σ(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A001065 – Sum of proper divisors

* **Base:** s(n)=σ(n)-n.
* **Bridge:** inherits structure from σ(n).
* **Emergence:** classifies numbers as perfect, abundant, or deficient.


In [ ]:
print("="*70)
print("A001065 – SUM OF PROPER DIVISORS")


In [ ]:
# --- s(n) Direct Computation and Classification ---
N=30
for n in range(1,N+1):
    sp = s_proper(n)
    if sp < n:
        tag = "deficient"
    elif sp > n:
        tag = "abundant"
    else:
        tag = "perfect"
    print(f"n={n}, s(n)={sp}, {tag}")



## Mathematical Expositor – A001065 – Sum of proper divisors

Explain the conceptual meaning of s(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A001221 – Number of distinct prime factors

* **Base:** counts distinct primes dividing n.
* **Bridge:** additive on coprimes; generating series via zeta identities.
* **Emergence:** normal order log log n (Hardy–Ramanujan).


In [ ]:
print("="*70)
print("A001221 – NUMBER OF DISTINCT PRIME FACTORS")


In [ ]:
# --- ω(n) Direct Computation ---
N=20
omega_vals=[omega(k) for k in range(1,N+1)]
print("ω(n) for n=1..20:", omega_vals)


In [ ]:
# --- ω(n) Additivity Check ---
# ω(ab) = ω(a)+ω(b) for gcd(a,b)=1
for a in range(1,11):
    for b in range(1,11):
        if gcd(a,b)==1:
            if omega(a*b) != omega(a) + omega(b):
                print("Additivity failed at", a, b)
print("Additivity holds for a,b ≤ 10")



## Mathematical Expositor – A001221 – Number of distinct prime factors

Explain the conceptual meaning of ω(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A001222 – Total number of prime factors

* **Base:** counts prime factors with multiplicity.
* **Bridge:** completely additive on all integers.
* **Emergence:** connects to Liouville function and normal orders.


In [ ]:
print("="*70)
print("A001222 – TOTAL NUMBER OF PRIME FACTORS")


In [ ]:
# --- Ω(n) Direct Computation ---
N=20
Omega_vals=[Omega(k) for k in range(1,N+1)]
print("Ω(n) for n=1..20:", Omega_vals)


In [ ]:
# --- Ω(n) Complete Additivity Check ---
for a in range(2,11):
    for b in range(2,11):
        if Omega(a*b) != Omega(a) + Omega(b):
            print("Complete additivity failed at",a,b)
print("Complete additivity holds for a,b ≤ 10")



## Mathematical Expositor – A001222 – Total number of prime factors

Explain the conceptual meaning of Ω(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A008683 – Möbius function

* **Base:** μ(n)=0 if n has squared prime factors; μ(n)=(-1)^k otherwise.
* **Bridge:** Dirichlet inverse of constant function; 1 * μ = ε.
* **Emergence:** Möbius inversion; connection to Riemann Hypothesis.


In [ ]:
print("="*70)
print("A008683 – MÖBIUS FUNCTION")


In [ ]:
# --- μ(n) Direct Computation ---
N=20
mu_vals=[mu(k) for k in range(1,N+1)]
print("μ(n) for n=1..20:", mu_vals)


In [ ]:
# --- μ(n) Multiplicativity Check ---
for a in range(1,11):
    for b in range(1,11):
        if gcd(a,b)==1:
            if mu(a*b) != mu(a)*mu(b):
                print("Multiplicativity failed at", a, b)
print("Multiplicativity holds for a,b ≤ 10")


In [ ]:
# --- μ(n) Convolution Check ---
# Check that sum_{d|n} μ(d) = 0 for n>1
for n in range(2,50):
    s = sum(mu(d) for d in range(1,n+1) if n % d == 0)
    if s != 0:
        print("Summatory μ failed at", n)
print("Summatory μ holds for n ≤ 49")


In [ ]:
# --- μ(n) Dirichlet Series Approximation ---
# Approximate 1/ζ(s) by sum μ(n)/n^s for s=2
s=2
partial_sum=sum(mu(n)/ (n**s) for n in range(1,2000))
import mpmath as mp
mp.dps=30
approx=1/mp.zeta(s)
print("Σ μ(n)/n^s (partial,2000 terms) ≈", partial_sum)
print("1/ζ(s) ≈", approx)



## Mathematical Expositor – A008683 – Möbius function

Explain the conceptual meaning of μ(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A000720 – Prime counting function

* **Base:** π(n) counts primes ≤ n.
* **Bridge:** approximated by n/log n; explicit formula via zeros of ζ(s).
* **Emergence:** prime number theorem and error terms.


In [ ]:
print("="*70)
print("A000720 – PRIME COUNTING FUNCTION")


In [ ]:
# --- π(n) Direct Computation ---
N=50
pi_vals=[pi_fn(k) for k in range(1,N+1)]
print("π(n) for n=1..50:", pi_vals)


In [ ]:
# --- π(n) Approximation Check ---
# Compare π(n) to n/log n for some values
for n in [10,20,50,100,200]:
    if n > 1:
        approx = n/log(n)
        print(f"n={n}, π(n)={pi_fn(n)}, n/log(n)={approx:.3f}, ratio={pi_fn(n)/approx:.3f}")



## Mathematical Expositor – A000720 – Prime counting function

Explain the conceptual meaning of π(n) in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.



## Sequence Analyst – Mechanisms & DAG for A002110 – Primorials

* **Base:** product of first n primes.
* **Bridge:** square‑free; contains all primes up to p_n.
* **Emergence:** Euclid’s proof; large prime gaps; Chebyshev ratio.


In [ ]:
print("="*70)
print("A002110 – PRIMORIALS")


In [ ]:
# --- Primorial Direct Computation ---
# Compute first 6 primorials
primorials=[primorial(k) for k in range(1,7)]
print("Primorials p_n# for n=1..6:", primorials)


In [ ]:
# --- Primorial + 1 Check ---
# Check primality of p_n# + 1 for first few n
for k in range(1,7):
    p = primorial(k)
    candidate = p + 1
    # test if candidate has prime factor > last prime used
    pf = prime_factors(candidate)
    print(f"n={k}, p_n#={p}, p_n#+1={candidate}, factors={pf}")



## Mathematical Expositor – A002110 – Primorials

Explain the conceptual meaning of p_n# in the context of arithmetic functions, multiplicativity, and analytic number theory. Discuss how the base definition leads to the bridge identities and what insights emerge from the analytic viewpoint.
